# 01 — Ingest Contoso Data

This notebook downloads the Contoso V2 parquet dataset and writes it as Delta tables into the Fabric Lakehouse (bronze layer).

**Prerequisites:**
- This notebook must be run inside a Fabric Lakehouse notebook environment
- The Lakehouse `contoso_lakehouse` must be attached

**What it does:**
1. Downloads `parquet-100k.7z` from GitHub
2. Extracts the parquet files to `/tmp/contoso/`
3. Reads each parquet file with **pandas** (not spark.read.parquet — see note below)
4. Converts to Spark DataFrame and writes Delta tables to the Lakehouse (bronze layer)

**Why pandas for reading?**
Fabric notebooks run Spark in a managed environment. `spark.read.parquet()` on local `/tmp` paths
triggers ABFS (Azure Blob Filesystem) auth even for local files, which fails. Reading with pandas
on the driver node bypasses this — pandas reads locally, then `spark.createDataFrame()` converts
in-memory before writing to the Lakehouse via `saveAsTable()`.

**Output tables:** `bronze_sales`, `bronze_customer`, `bronze_product`, `bronze_store`, `bronze_date`, `bronze_currency_exchange`

In [ ]:
# Step 1: Download Contoso parquet-100k dataset
import subprocess

DOWNLOAD_URL = "https://github.com/sql-bi/Contoso-Data-Generator-V2-data/releases/download/ready-to-use-data/parquet-100k.7z"
DOWNLOAD_PATH = "/tmp/contoso-parquet-100k.7z"
EXTRACT_PATH = "/tmp/contoso/"

print(f"Downloading from: {DOWNLOAD_URL}")
subprocess.run(["wget", "-q", "-O", DOWNLOAD_PATH, DOWNLOAD_URL], check=True)
print("Download complete.")

In [ ]:
# Step 2: Extract the 7z archive
import os

os.makedirs(EXTRACT_PATH, exist_ok=True)

# Install 7zip if not available
subprocess.run(["apt-get", "install", "-y", "-q", "p7zip-full"], check=True)
subprocess.run(["7z", "x", DOWNLOAD_PATH, f"-o{EXTRACT_PATH}", "-y"], check=True)

print("Extraction complete. Files:")
for f in sorted(os.listdir(EXTRACT_PATH)):
    print(f"  {f}")

In [ ]:
# Step 3: Load each parquet file using the pandas-bridge pattern and write as Delta table
#
# IMPORTANT: Do NOT use spark.read.parquet() for local /tmp files.
# Even with a local path, Spark routes through ABFS which fails without storage credentials.
# Fix: read with pandas (driver-local, no auth needed), convert to Spark, then saveAsTable.
#
# Note: the archive extracts without underscores — currencyexchange.parquet (not currency_exchange)

import pandas as pd
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

# Mapping: Delta table name -> parquet filename
TABLES = {
    "bronze_sales":             "sales.parquet",
    "bronze_customer":          "customer.parquet",
    "bronze_product":           "product.parquet",
    "bronze_store":             "store.parquet",
    "bronze_date":              "date.parquet",
    "bronze_currency_exchange": "currencyexchange.parquet",
}

for target_table, filename in TABLES.items():
    parquet_path = os.path.join(EXTRACT_PATH, filename)

    if not os.path.exists(parquet_path):
        print(f"WARNING: Skipping {target_table} - file not found: {parquet_path}")
        continue

    print(f"Loading {filename} -> {target_table} ...")

    # Read locally with pandas - no ABFS auth required
    pdf = pd.read_parquet(parquet_path)

    # Convert to Spark DataFrame
    df = spark.createDataFrame(pdf)

    # Write as Delta table to Lakehouse (requires notebook context, not Livy)
    df.write.format("delta").mode("overwrite").saveAsTable(target_table)

    row_count = spark.table(target_table).count()
    print(f"  OK {target_table}: {row_count:,} rows")

print("\nIngestion complete! Bronze tables are ready.")

In [ ]:
# Step 4: Verify - show row counts for all bronze tables
print("Bronze table row counts:")
for target_table in TABLES.keys():
    try:
        count = spark.table(target_table).count()
        print(f"  {target_table}: {count:,}")
    except Exception as e:
        print(f"  {target_table}: WARNING {e}")

In [ ]:
# Step 5: Inspect continent values for serving layer reference
# Note: bronze columns are CamelCase - use 'Continent' not 'continent'
print("Distinct Continent values in bronze_customer:")
spark.table("bronze_customer").select("Continent").distinct().orderBy("Continent").show()